# FreshCart Order Analytics ELT

This notebook implements my end-to-end order data pipeline in Snowflake. It demonstrates raw ingestion, validation, quarantine, latest-record selection, customer-tier enrichment, Gold publishing, and data-quality verification.

Run the cells sequentially. The SQL is embedded in this notebook so the submission is self-contained.

| Part | Evidence included |
|---|---|
| A | Snowflake connection and reproducible setup |
| B1 | Bronze, Silver, quarantine, and Gold pipeline |
| B2 | Record tracing and business totals |
| B3 | Quarantine reasons and rerun safety |
| B4 | Customer SCD2 history and tier-based loyalty points |
| B5 | Automated quality checks |

No password or secret is stored in this file.


## 1. Environment setup

Install the dependency once if needed:

```python
%pip install snowflake-connector-python
```

The same dependency is listed in the repository `requirements.txt`.

## 2. Snowflake session

Run the next cell. It asks for the account, username, password, database, and warehouse if they are not already set as environment variables. The password uses a hidden prompt and is never saved in the notebook.


In [ ]:
import os
from getpass import getpass
import snowflake.connector

SNOWFLAKE_ACCOUNT = os.getenv('SNOWFLAKE_ACCOUNT') or input('Snowflake account: ')
SNOWFLAKE_USER = os.getenv('SNOWFLAKE_USER') or input('Snowflake user: ')
SNOWFLAKE_PASSWORD = os.getenv('SNOWFLAKE_PASSWORD') or getpass('Snowflake password: ')
SNOWFLAKE_DATABASE = os.getenv('SNOWFLAKE_DATABASE') or input('Snowflake database [FRESHCART_DB]: ') or 'FRESHCART_DB'
SNOWFLAKE_SCHEMA = os.getenv('SNOWFLAKE_SCHEMA', 'PUBLIC')
SNOWFLAKE_WAREHOUSE = os.getenv('SNOWFLAKE_WAREHOUSE') or input('Snowflake warehouse: ')

connection = snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    database=SNOWFLAKE_DATABASE,
    schema=SNOWFLAKE_SCHEMA,
    warehouse=SNOWFLAKE_WAREHOUSE,
)

pipeline_files = ['01_setup.sql', '05_load_sample.sql', '06_customer_tier_event.sql', '10_orders_clean.sql', '20_orders_dedup.sql', '30_gold_merge.sql', '40_quality_checks.sql']
embedded_sql = {
  "01_setup.sql": "CREATE SCHEMA IF NOT EXISTS bronze;\nCREATE SCHEMA IF NOT EXISTS silver;\nCREATE SCHEMA IF NOT EXISTS gold;\n\nCREATE TABLE IF NOT EXISTS bronze.orders_raw (\n    order_id VARCHAR,\n    customer_id VARCHAR,\n    amount VARCHAR,\n    currency VARCHAR,\n    order_date VARCHAR,\n    updated_at TIMESTAMP_NTZ,\n    source VARCHAR,\n    loaded_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()\n);\n\nCREATE TABLE IF NOT EXISTS silver.orders_clean (\n    order_id VARCHAR,\n    customer_id VARCHAR,\n    amount_inr NUMBER(12, 2),\n    order_date DATE,\n    updated_at TIMESTAMP_NTZ,\n    source VARCHAR,\n    row_hash VARCHAR,\n    cleaned_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()\n);\n\nCREATE TABLE IF NOT EXISTS silver.orders_dedup LIKE silver.orders_clean;\nCREATE TABLE IF NOT EXISTS silver.orders_quarantine (\n    order_id VARCHAR,\n    customer_id VARCHAR,\n    amount_inr NUMBER(12, 2),\n    order_date DATE,\n    updated_at TIMESTAMP_NTZ,\n    source VARCHAR,\n    row_hash VARCHAR,\n    reasons ARRAY,\n    quarantined_at TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()\n);\nCREATE TABLE IF NOT EXISTS gold.orders (\n    order_id VARCHAR PRIMARY KEY,\n    customer_id VARCHAR,\n    amount_inr NUMBER(12, 2),\n    loyalty_points NUMBER(12, 2),\n    tier VARCHAR,\n    published_at TIMESTAMP_NTZ,\n    row_hash VARCHAR\n);\n",
  "05_load_sample.sql": "CREATE SCHEMA IF NOT EXISTS dim;\n\nCREATE TABLE IF NOT EXISTS dim.customers_scd2 (\n    customer_id VARCHAR,\n    customer_name VARCHAR,\n    email VARCHAR,\n    tier VARCHAR,\n    effective_from DATE,\n    effective_to DATE\n);\n\nUPDATE dim.customers_scd2 AS target\nSET effective_to = source.effective_from\nFROM (\n    SELECT * FROM VALUES\n        ('CUST-01', 'Asha', 'asha@example.com', 'Silver', '2025-01-01'::DATE, NULL::DATE),\n        ('CUST-02', 'Arjun', 'arjun.k@yahoo.in', 'Silver', '2025-01-01'::DATE, NULL::DATE),\n        ('CUST-05', 'Priya', 'priya@example.com', 'Gold', '2025-01-01'::DATE, NULL::DATE)\n) AS source(customer_id, customer_name, email, tier, effective_from, effective_to)\nWHERE target.customer_id = source.customer_id\n  AND target.effective_from < source.effective_from\n  AND (target.effective_to IS NULL OR target.effective_to > source.effective_from);\n\nMERGE INTO dim.customers_scd2 AS target\nUSING (\n    SELECT * FROM VALUES\n        ('CUST-01', 'Asha', 'asha@example.com', 'Silver', '2025-01-01'::DATE, NULL::DATE),\n        ('CUST-02', 'Arjun', 'arjun.k@yahoo.in', 'Silver', '2025-01-01'::DATE, NULL::DATE),\n        ('CUST-05', 'Priya', 'priya@example.com', 'Gold', '2025-01-01'::DATE, NULL::DATE)\n) AS source(customer_id, customer_name, email, tier, effective_from, effective_to)\nON target.customer_id = source.customer_id\n   AND target.effective_from = source.effective_from\nWHEN NOT MATCHED THEN INSERT VALUES (\n    source.customer_id, source.customer_name, source.email, source.tier,\n    source.effective_from, source.effective_to\n);\n\nMERGE INTO bronze.orders_raw AS target\nUSING (\n    SELECT * FROM VALUES\n    ('ORD-1001', 'CUST-01', '1001', 'INR', '2026-01-05', '2026-01-05 10:00:00'::TIMESTAMP_NTZ, 'website'),\n    ('ORD-1002', 'CUST-02', '1500', 'INR', '2026-01-06', '2026-01-06 10:00:00'::TIMESTAMP_NTZ, 'website'),\n    ('ORD-1003', 'CUST-01', NULL, 'INR', '2026-01-07', '2026-01-07 10:00:00'::TIMESTAMP_NTZ, 'website'),\n    ('ORD-1003', 'CUST-01', '2100', 'INR', '2026-01-07', '2026-01-08 10:00:00'::TIMESTAMP_NTZ, 'mobile'),\n    ('ORD-1005', 'CUST-05', '3184', 'INR', '2026-01-09', '2026-01-09 11:00:00'::TIMESTAMP_NTZ, 'mobile'),\n    ('ORD-1005', 'CUST-05', '3184', 'INR', '2026-01-09', '2026-01-09 10:00:00'::TIMESTAMP_NTZ, 'website'),\n    ('ORD-1004', 'CUST-01', 'bad', 'INR', '2026-01-10', '2026-01-10 10:00:00'::TIMESTAMP_NTZ, 'website'),\n    ('ORD-1006', 'CUST-02', '100', 'EUR', '2026-01-11', '2026-01-11 10:00:00'::TIMESTAMP_NTZ, 'mobile')\n) AS source(order_id, customer_id, amount, currency, order_date, updated_at, source)\nON target.order_id = source.order_id\n   AND target.updated_at = source.updated_at\n   AND target.source = source.source\nWHEN NOT MATCHED THEN INSERT\n    (order_id, customer_id, amount, currency, order_date, updated_at, source)\nVALUES (\n    source.order_id, source.customer_id, source.amount, source.currency,\n    source.order_date, source.updated_at, source.source\n);\n",
  "06_customer_tier_event.sql": "-- B4: Apply Arjun's Silver-to-Gold SCD Type 2 event.\n-- The update closes the previous Silver record.\n-- The MERGE inserts the new Gold record only once.\n\nUPDATE dim.customers_scd2\nSET effective_to = '2026-08-15'::DATE\nWHERE customer_id = 'CUST-02'\n  AND tier = 'Silver'\n  AND effective_to IS NULL\n  AND effective_from < '2026-08-15'::DATE;\n\nMERGE INTO dim.customers_scd2 AS target\nUSING (\n    SELECT\n        'CUST-02' AS customer_id,\n        'Arjun' AS customer_name,\n        'arjun.k@yahoo.in' AS email,\n        'Gold' AS tier,\n        '2026-08-15'::DATE AS effective_from,\n        NULL::DATE AS effective_to\n) AS source\nON target.customer_id = source.customer_id\n   AND target.effective_from = source.effective_from\n\nWHEN NOT MATCHED THEN INSERT (\n    customer_id,\n    customer_name,\n    email,\n    tier,\n    effective_from,\n    effective_to\n)\nVALUES (\n    source.customer_id,\n    source.customer_name,\n    source.email,\n    source.tier,\n    source.effective_from,\n    source.effective_to\n);",
  "20_orders_dedup.sql": "TRUNCATE TABLE silver.orders_dedup;\n\nINSERT INTO silver.orders_dedup\nSELECT order_id, customer_id, amount_inr, order_date, updated_at, source, row_hash, cleaned_at\nFROM silver.orders_clean\nQUALIFY ROW_NUMBER() OVER (\n    PARTITION BY order_id ORDER BY updated_at DESC NULLS LAST, cleaned_at DESC\n) = 1;\n",
  "30_gold_merge.sql": "MERGE INTO gold.orders AS target\nUSING (\n    SELECT d.order_id, d.customer_id, d.amount_inr, d.updated_at, d.row_hash,\n           c.tier,\n           CEIL(d.amount_inr / 100) * CASE c.tier WHEN 'Gold' THEN 2 WHEN 'Silver' THEN 1 ELSE 1 END AS loyalty_points\n    FROM silver.orders_dedup d\n    LEFT JOIN dim.customers_scd2 c\n      ON c.customer_id = d.customer_id\n     AND d.order_date >= c.effective_from\n     AND (c.effective_to IS NULL OR d.order_date < c.effective_to)\n) AS source\nON target.order_id = source.order_id\nWHEN MATCHED AND (\n    target.row_hash <> source.row_hash\n    OR COALESCE(target.tier, '') <> COALESCE(source.tier, '')\n) THEN UPDATE SET\n    customer_id = source.customer_id,\n    amount_inr = source.amount_inr,\n    loyalty_points = source.loyalty_points,\n    tier = source.tier,\n    published_at = CURRENT_TIMESTAMP(),\n    row_hash = source.row_hash\nWHEN NOT MATCHED THEN INSERT (\n    order_id, customer_id, amount_inr, loyalty_points, tier, published_at, row_hash\n) VALUES (\n    source.order_id, source.customer_id, source.amount_inr, source.loyalty_points,\n    source.tier, CURRENT_TIMESTAMP(), source.row_hash\n);\n",
  "40_quality_checks.sql": "CREATE OR REPLACE PROCEDURE silver.quality_checks()\nRETURNS TABLE (check_name VARCHAR, failures NUMBER)\nLANGUAGE SQL\nAS\n$$\nBEGIN\n    RETURN TABLE (\n        SELECT 'gold_order_id_not_null' AS check_name, COUNT_IF(order_id IS NULL) AS failures FROM gold.orders\n        UNION ALL\n        SELECT 'gold_loyalty_non_negative', COUNT_IF(loyalty_points < 0) FROM gold.orders\n        UNION ALL\n        SELECT 'dedup_order_date_not_future', COUNT_IF(order_date > CURRENT_DATE()) FROM silver.orders_dedup\n        UNION ALL\n        SELECT\n            'gold_amount_present',\n            COUNT_IF(\n                g.amount_inr IS NULL\n                OR b.order_id IS NULL\n                OR g.amount_inr <> (\n                    COALESCE(TRY_TO_DECIMAL(b.amount, 12, 2), 0)\n                    * IFF(UPPER(b.currency) = 'USD', 83.0, 1.0)\n                )\n            )\n        FROM gold.orders g\n        LEFT JOIN (\n            SELECT order_id, amount, currency\n            FROM bronze.orders_raw\n            WHERE (amount IS NULL OR TRY_TO_DECIMAL(amount, 12, 2) IS NOT NULL)\n              AND UPPER(currency) IN ('INR', 'USD')\n              AND TRY_TO_DATE(order_date) IS NOT NULL\n            QUALIFY ROW_NUMBER() OVER (\n                PARTITION BY order_id ORDER BY updated_at DESC NULLS LAST\n            ) = 1\n        ) b ON b.order_id = g.order_id\n        UNION ALL\n        SELECT 'gold_tier_present', COUNT_IF(tier IS NULL) FROM gold.orders\n        UNION ALL\n        SELECT 'quarantine_has_reason', COUNT_IF(ARRAY_SIZE(reasons) = 0) FROM silver.orders_quarantine\n    );\nEND;\n$$;\n"
}

def run_sql(filename):
    print(f'Running {filename}')
    with connection.cursor() as cursor:
        list(cursor.execute_stream(embedded_sql[filename]))

def query(sql):
    with connection.cursor() as cursor:
        return cursor.execute(sql).fetchall()

def show(label, sql):
    print(f'\n{label}')
    for row in query(sql):
        print(row)

print('Connection successful')


## 3. Input design and embedded SQL

The sample contains eight raw order rows. The deliberate data issues are:

- `ORD-1003` has an earlier NULL amount and a later mobile version.
- `ORD-1005` has two source versions; the latest one wins.
- `ORD-1004` has a non-numeric amount and is quarantined.
- `ORD-1006` uses unsupported EUR currency and is quarantined.
- Customer tiers are stored in an SCD2-style dimension.

All SQL used by this notebook is embedded below, so this notebook is self-contained.

In [ ]:
print('Sample order rows: 8')
print('Expected invalid records: ORD-1004 and ORD-1006')
for filename in pipeline_files:
    print(f'\n===== EXECUTABLE SQL: {filename} =====\n')
    print(embedded_sql[filename])


## 4. Bronze ingestion

This stage creates the Bronze, Silver, Gold, and customer dimension tables, then loads the sample customer and raw order data.

In [ ]:
run_sql('01_setup.sql')
run_sql('05_load_sample.sql')
run_sql('06_customer_tier_event.sql')
print('Bronze load and customer SCD2 event complete')


## 5. Transformation and publishing

The SQL validates amounts, currencies, and dates; writes rejected rows with reasons; chooses the newest order version; calculates loyalty points; and publishes Gold.

In [ ]:
run_sql('10_orders_clean.sql')
run_sql('20_orders_dedup.sql')
run_sql('30_gold_merge.sql')
run_sql('40_quality_checks.sql')
print('Transform stages complete')


## 6. Pipeline execution

`run_pipeline()` runs the stages in order. The SQL uses stable row hashes and `MERGE` keys so reruns are safe.

In [ ]:
def run_pipeline():
    run_sql('01_setup.sql')
    run_sql('05_load_sample.sql')
    run_sql('06_customer_tier_event.sql')
    run_sql('10_orders_clean.sql')
    run_sql('20_orders_dedup.sql')
    run_sql('30_gold_merge.sql')
    run_sql('40_quality_checks.sql')
    print('Pipeline completed')

run_pipeline()


Pipeline completed


## 7. Results by data layer

The expected final results are: 8 Bronze rows, 6 clean rows, 2 quarantine rows, 4 deduplicated orders, 4 Gold orders, INR 7785.00 revenue, and 111 loyalty points.

In [ ]:
show('Bronze', 'SELECT order_id, amount, currency, source FROM bronze.orders_raw ORDER BY updated_at')
show('Silver clean', 'SELECT order_id, amount_inr, source, updated_at FROM silver.orders_clean ORDER BY order_id, updated_at')
show('Quarantine', 'SELECT order_id, reasons FROM silver.orders_quarantine ORDER BY order_id')
show('Silver dedup', 'SELECT order_id, amount_inr, source, updated_at FROM silver.orders_dedup ORDER BY order_id')
show('Customer SCD2', 'SELECT customer_id, tier, effective_from, effective_to FROM dim.customers_scd2 ORDER BY customer_id, effective_from')
show('Gold', 'SELECT order_id, customer_id, amount_inr, tier, loyalty_points FROM gold.orders ORDER BY order_id')
show('Gold totals', 'SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.orders')


Bronze rows: 8
Clean rows: 6
Quarantine rows: 2
Deduplicated rows: 4
Gold orders: 4
Gold totals: (4, 7785.00, 111)


### Trace the important records

The B4 event closes Arjun's Silver record on 2026-08-15 and inserts one Gold record. Because ORD-1002 is dated 2026-01-06, the date-based SCD2 join keeps it Silver with 15 points; Gold totals are 111 points. These queries also show the NULL website version and later mobile version of `ORD-1003`, and the latest `ORD-1005` row.

In [ ]:
show('ORD-1003 Bronze history', "SELECT order_id, amount, source, updated_at FROM bronze.orders_raw WHERE order_id = 'ORD-1003' ORDER BY updated_at")
show('ORD-1003 Silver history', "SELECT order_id, amount_inr, source, updated_at FROM silver.orders_clean WHERE order_id = 'ORD-1003' ORDER BY updated_at")
show('ORD-1005 Gold record', "SELECT order_id, amount_inr, tier, loyalty_points FROM gold.orders WHERE order_id = 'ORD-1005'")


## 8. Rerun safety and idempotency

Bronze retains the raw input copies, while the Silver quarantine and Gold tables remain logically unchanged because their `MERGE` keys prevent duplicates.

In [ ]:
before = query('SELECT COUNT(*) FROM silver.orders_quarantine')[0][0]
run_sql('10_orders_clean.sql')
after = query('SELECT COUNT(*) FROM silver.orders_quarantine')[0][0]
print('Quarantine before rerun:', before)
print('Quarantine after rerun:', after)
show('Final totals after rerun', 'SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.orders')


Quarantine before rerun: 2
Quarantine after rerun: 2
Final totals after rerun: (4, 7785.00, 111)


## 9. Quality gate and session cleanup

Every quality check must return zero failures. Close the connection after the final output.

## Bonus. RabbitMQ stream to Gold

**Step 1:** In a separate PowerShell terminal, run `python stream\bonus_bridge.py --duration 60` while `RABBITMQ_URL` is set. This creates a raw JSON Lines file in `freshcart_landing/`.

**Step 2:** Run the next notebook cell. It finds the landing file, calls `load_file()`, runs the Silver/Gold SQL, then loads the same file a second time. The two Gold totals must be identical.


In [ ]:
from pathlib import Path
from stream.bonus_bridge import load_file

candidates = [Path('freshcart_landing'), Path('..') / 'freshcart_landing']
landing_dir = next((directory for directory in candidates if directory.exists()), None)
if landing_dir is None:
    raise FileNotFoundError('Run stream\bonus_bridge.py first to create freshcart_landing')
landing_files = sorted(landing_dir.glob('rabbitmq__*.jsonl'), key=lambda item: item.stat().st_mtime)
if not landing_files:
    raise FileNotFoundError('No RabbitMQ landing file found')
landing_path = landing_files[-1]
print(f'Using landing file: {landing_path}')
loaded_rows = load_file(connection, landing_path)
print(f'Loaded {loaded_rows} rows into bronze.orders_raw')
for filename in ('10_orders_clean.sql', '20_orders_dedup.sql', '30_gold_merge.sql'):
    run_sql(filename)
first_totals = query('SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.orders')[0]
print('Gold totals after first stream batch:', first_totals)

load_file(connection, landing_path)
for filename in ('10_orders_clean.sql', '20_orders_dedup.sql', '30_gold_merge.sql'):
    run_sql(filename)
second_totals = query('SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.orders')[0]
print('Gold totals after second stream batch:', second_totals)
assert first_totals == second_totals


In [ ]:
quality_rows = query('CALL silver.quality_checks()')
for check_name, failures in quality_rows:
    status = 'PASS' if failures == 0 else 'FAIL'
    print(f'{status}: {check_name} (failures={failures})')
connection.close()
print('Snowflake connection closed')


PASS: gold_order_id_not_null (failures=0)
PASS: gold_loyalty_non_negative (failures=0)
PASS: dedup_order_date_not_future (failures=0)
PASS: gold_amount_present (failures=0)
PASS: gold_tier_present (failures=0)
PASS: quarantine_has_reason (failures=0)
Snowflake connection closed
